In [29]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, auc, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorboard.backend.event_processing import event_accumulator
from torch.amp import GradScaler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from transformers import AutoConfig, AutoFeatureExtractor, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.simplefilter("ignore", UserWarning)

import torchaudio
import random
import pandas as pd

import torchaudio
import random
import pandas as pd
import math

def generate_random_segments(
    file_list,
    sr=16000,
    min_sec=0.4,
    max_sec=0.6,
    min_samples=4000
):
    records = []

    for path in tqdm(file_list):
        wav, fs = torchaudio.load(path)
        if fs != sr:
            wav = torchaudio.functional.resample(wav, fs, sr)

        wav = wav.squeeze(0)
        length = wav.shape[0]

        if length < min_samples:
            continue

        audio_sec = length / sr

        # Pick a duration once for consistency per file
        dur = random.uniform(min_sec, max_sec)
        seg_len = int(dur * sr)

        # Not enough to produce even one segment above min_samples
        if seg_len < min_samples:
            records.append({
                "path_file": path,
                "index_start": 0,
                "index_end": length,
            })
            continue

        # Dynamic repeat count
        repeat = max(1, math.floor(audio_sec / dur))

        for _ in range(repeat):
            if seg_len >= length:
                start = 0
            else:
                start = random.randint(0, length - seg_len)

            end = start + seg_len

            records.append({
                "path_file": path,
                "index_start": start,
                "index_end": end,
            })

    return pd.DataFrame(records)



/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough


In [12]:
df_cough_train = pd.read_csv('/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.train')
df_cough_test = pd.read_csv('/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.val')
df_cough_train = df_cough_train[['path_file']]
df_cough_test = df_cough_test[['path_file']]

df_noise = pd.read_csv("/run/media/fourier/Data1/Pras/Interspeech2025/RIRS_NOISES/data_augmentation_noises_labels.tsv", delimiter="\t", header=None)
df_noise_train, df_noise_test = train_test_split(df_noise, test_size=0.2)
df_noise_train = generate_random_segments(df_noise_train[0].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.6)
df_noise_test = generate_random_segments(df_noise_test[0].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.6)
df_noise_train = df_noise_train.sample(n=8058)
df_noise_test = df_noise_test.sample(n=1146)

df_speech_train = pd.read_csv("/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/metadata.csv.train")
df_speech_test = pd.read_csv("/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/metadata.csv.dev")
df_speech_train["filepath"] = "/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/" + df_speech_train["filepath"]
df_speech_test["filepath"] = "/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/" + df_speech_test["filepath"]
df_speech_train = generate_random_segments(df_speech_train['filepath'].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.6)
df_speech_test = generate_random_segments(df_speech_test['filepath'].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.6)
df_speech_train = df_speech_train.sample(n=8058)
df_speech_test = df_speech_test.sample(n=1146)

df_cough_train = df_cough_train.reset_index(drop=True)
df_cough_test = df_cough_test.reset_index(drop=True)

df_speech_train = df_speech_train.reset_index(drop=True)
df_speech_test = df_speech_test.reset_index(drop=True)

df_noise_train = df_noise_train.reset_index(drop=True)
df_noise_test = df_noise_test.reset_index(drop=True)

df_cough_train['index_start'] = 0
df_cough_train['index_end'] = -1
df_cough_train['label'] = 1
df_cough_test['index_start'] = 0
df_cough_test['index_end'] = -1
df_cough_test['label'] = 1

df_noise_train['label'] = 0
df_noise_test['label'] = 0
df_speech_train['label'] = 0
df_speech_test['label'] = 0

df_train = pd.concat([df_cough_train, df_speech_train, df_noise_train], axis=0, ignore_index=True)
df_test = pd.concat([df_cough_test, df_speech_test, df_noise_test], axis=0, ignore_index=True)

In [15]:
df_train['label'].value_counts()

label
0    16116
1     8058
Name: count, dtype: int64

In [16]:
df_train.to_csv(f"/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cough_detection.csv.train", index=False)
df_test.to_csv(f"/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cough_detection.csv.test", index=False)

# Build for Wavlm vs Cough

In [1]:
import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, auc, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorboard.backend.event_processing import event_accumulator
from torch.amp import GradScaler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from transformers import AutoConfig, AutoFeatureExtractor, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split

import warnings
warnings.simplefilter("ignore", UserWarning)

def filter_participants(df, participant_col, file_col, min_files=5, max_files=15, seed=42):
    """
    Normalize participant-level sample counts.
    - Remove participants with < min_files
    - Randomly downsample participants with > max_files
    - Keep only participants within the [min_files, max_files] band
    """
    # Compute counts
    counts = df.groupby(participant_col)[file_col].count()

    # Identify cohorts
    drop_ids = counts[counts < min_files].index
    downsample_ids = counts[counts > max_files].index

    # Filter low-volume participants
    df_filtered = df[~df[participant_col].isin(drop_ids)]

    # Downsample oversized participants
    output_frames = []
    for pid, group in df_filtered.groupby(participant_col):
        if pid in downsample_ids:
            group = group.sample(n=max_files, random_state=seed)
        output_frames.append(group)

    return pd.concat(output_frames, ignore_index=True)


def balance_participants(
    df,
    participant_col="participant",
    label_col="disease_status",
    ratio=1.0,
    seed=42,
    minority_label=None,
):
    """
    Balances the dataset by sampling participants, not individual rows.

    minority_label:
        None → auto-detect minority class based on participant counts
        0 or 1 → force a specific class to be treated as minority
    """

    # Participant → label mapping
    p_map = df.groupby(participant_col)[label_col].first()

    # Participant counts per class
    counts = p_map.value_counts()

    # Determine minority size
    if minority_label is None:
        minority_size = counts.min()
    else:
        # Forced minority: its size = actual participant count of that label
        minority_size = counts[minority_label]

    # Cap for majority classes
    majority_cap = int(minority_size * ratio)

    keep_pids = []

    for label in counts.index:
        pids = p_map[p_map == label]

        if label == minority_label:
            # Keep minority participants fully
            keep_pids.extend(pids.index)
        else:
            # Apply capping to majority groups
            if len(pids) > majority_cap:
                pids = pids.sample(majority_cap, random_state=seed)
            keep_pids.extend(pids.index)

    return df[df[participant_col].isin(keep_pids)].reset_index(drop=True)

def sample_rows_by_participant(df, target_rows=3042, participant_col="participant", label_col="disease_status", random_state=42):
    """
    Sample rows from a dataframe per participant while roughly balancing classes.

    Args:
        df (pd.DataFrame): DataFrame containing participants and labels.
        target_rows (int): Approximate total number of rows to sample.
        participant_col (str): Column name for participant IDs.
        label_col (str): Column name for labels.
        random_state (int): Seed for reproducibility.

    Returns:
        pd.DataFrame: Sampled dataframe.
    """
    # Group participants by their label
    participants = df.groupby(participant_col)[label_col].first().reset_index()

    # Shuffle participants per class
    classes = participants[label_col].unique()
    class_participants = {}
    for c in classes:
        class_participants[c] = participants[participants[label_col] == c].sample(frac=1, random_state=random_state)

    # Helper to select participants until target rows reached
    def select_participants(part_df, target_rows):
        selected = []
        total_rows = 0
        for p in part_df[participant_col]:
            part_rows = len(df[df[participant_col] == p])
            if total_rows + part_rows > target_rows:
                break
            selected.append(p)
            total_rows += part_rows
        return selected

    # Roughly split target rows evenly across classes
    rows_per_class = target_rows // len(classes)
    selected_participants = []
    for c in classes:
        selected_participants += select_participants(class_participants[c], rows_per_class)

    # Filter original dataframe
    df_sampled = df[df[participant_col].isin(selected_participants)]

    return df_sampled


def assign_split_column(
    df,
    participant_col="participant",
    label_col="disease_status",
    train_ratio=0.7,  # adjusted to leave space for val
    val_ratio=0.1,    # fraction of participants for validation
    seed=42
):
    """
    Stratified participant split into train/val/test.
    Guarantees:
        • No participant overlap
        • Class-wise train/val/test ratio per participant
        • Approximate row-level ratios
    """
    # Participant → label
    p_map = df.groupby(participant_col)[label_col].first()

    train_pids = []
    val_pids = []
    test_pids = []

    rng = np.random.default_rng(seed)

    for label in p_map.unique():
        pids = p_map[p_map == label].index
        n_total = len(pids)
        
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        
        # shuffle participants
        shuffled = rng.permutation(pids)
        train_sel = shuffled[:n_train]
        val_sel = shuffled[n_train:n_train+n_val]
        test_sel = shuffled[n_train+n_val:]

        train_pids.extend(train_sel)
        val_pids.extend(val_sel)
        test_pids.extend(test_sel)

    # assign split
    df = df.copy()
    df["split"] = "train"
    df.loc[df[participant_col].isin(val_pids), "split"] = "val"
    df.loc[df[participant_col].isin(test_pids), "split"] = "test"

    return df


def encode_participants(df, participant_col="participant"):
    """
    Encode participants as numeric speaker IDs.

    Args:
        df (pd.DataFrame): DataFrame containing a participant column.
        participant_col (str): Column name for participant IDs.
        new_col (str): Name of the new numeric column.

    Returns:
        pd.DataFrame: DataFrame with an additional numeric speaker_id column.
    """
    df = df.copy()
    participant_mapping = {p: i for i, p in enumerate(df[participant_col].unique())}
    df[participant_col] = df[participant_col].map(participant_mapping)
    return df

In [22]:
df_longi = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/longitudinal_original.csv')
df_longi = df_longi.rename(columns={"tb_status": "disease_status"})
df_longi = filter_participants(df_longi, "participant", "path_file", min_files=5, max_files=20, seed=42)
#df_longi = balance_participants(df_longi, participant_col="participant", label_col="disease_status", ratio=1.0, minority_label=1, seed=42)
df_longi = assign_split_column(df_longi, train_ratio=0.8, val_ratio=0.0)
df_longi["db"] = 0
df_longi = df_longi[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_longi['disease_status'].value_counts())

df_solic = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/solicited_original.csv')
df_solic = df_solic.rename(columns={"tb_status": "disease_status"})
df_solic = filter_participants(df_solic, "participant", "path_file", min_files=5, max_files=20, seed=42)
#df_solic = balance_participants(df_solic, participant_col="participant", label_col="disease_status", ratio=1.0, minority_label=1, seed=42)
df_solic = assign_split_column(df_solic, train_ratio=0.8, val_ratio=0.0)
df_solic["db"] = 1
df_solic = df_solic[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_solic['disease_status'].value_counts())

df_tbscreen_longi = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_longitudinal.csv')
df_tbscreen_longi = filter_participants(df_tbscreen_longi, "participant", "path_file", min_files=5, max_files=20, seed=42)
#df_tbscreen_longi = balance_participants(df_tbscreen_longi, participant_col="participant", label_col="disease_status", ratio=1.0, minority_label=1, seed=42)
df_tbscreen_longi = assign_split_column(df_tbscreen_longi, train_ratio=0.8, val_ratio=0.0)
df_tbscreen_longi["db"] = 2
df_tbscreen_longi = df_tbscreen_longi[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_tbscreen_longi['disease_status'].value_counts())

df_tbscreen_solic = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_solicited.csv')
df_tbscreen_solic = filter_participants(df_tbscreen_solic, "participant", "path_file", min_files=5, max_files=20, seed=42)
#df_tbscreen_solic = balance_participants(df_tbscreen_solic, participant_col="participant", label_col="disease_status", ratio=1.0, minority_label=1, seed=42)
df_tbscreen_solic = assign_split_column(df_tbscreen_solic, train_ratio=0.8, val_ratio=0.0)
df_tbscreen_solic["db"] = 3
df_tbscreen_solic = df_tbscreen_solic[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
#print(df_tbscreen_solic['disease_status'].value_counts())

df_cirdz = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/cirdz/metadata_wavs.csv')
df_cirdz = df_cirdz.rename(columns={"ground_truth_tb": "disease_status", "anon_id": "participant"})
df_cirdz = filter_participants(df_cirdz, "participant", "path_file", min_files=5, max_files=20, seed=42)
#df_cirdz = balance_participants(df_cirdz, participant_col="participant", label_col="disease_status", ratio=1.0, minority_label=1, seed=42)
df_cirdz = assign_split_column(df_cirdz, train_ratio=0.8, val_ratio=0.0)
df_cirdz["db"] = 4
df_cirdz = df_cirdz[['path_file', 'participant', 'sex', 'disease_status', 'split', 'db']]
df_cirdz['path_file'] = "/run/media/fourier/Data1/Pras/DatabaseLLM/cirdz/" + df_cirdz['path_file']
df_cirdz['sex'] = df_cirdz['sex'].replace({'m': 0, 'f': 1})
#print(df_longi['disease_status'].value_counts())

df_coughs = pd.concat([df_longi, df_solic, df_tbscreen_solic, df_tbscreen_longi, df_cirdz], axis=0, ignore_index=True)
df_coughs['disease_status'].value_counts()
print(df_coughs[df_coughs['split'] == 'train']['disease_status'].value_counts())
print(df_coughs[df_coughs['split'] == 'val']['disease_status'].value_counts())
print(df_coughs[df_coughs['split'] == 'test']['disease_status'].value_counts())

disease_status
0    14453
1     6690
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
disease_status
0    3674
1    1689
Name: count, dtype: int64


/tmp/ipykernel_1136876/3149036215.py:43: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cirdz['sex'] = df_cirdz['sex'].replace({'m': 0, 'f': 1})


In [23]:
df_cough_train = df_coughs[df_coughs['split'] == 'train'].reset_index(drop=True)
df_cough_test = df_coughs[df_coughs['split'] == 'test'].reset_index(drop=True)

In [ ]:
df_noise = pd.read_csv("/run/media/fourier/Data1/Pras/Interspeech2025/RIRS_NOISES/data_augmentation_noises_labels.tsv", delimiter="\t", header=None)
df_noise_train, df_noise_test = train_test_split(df_noise, test_size=0.2)
df_noise_train = generate_random_segments(df_noise_train[0].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.55)
df_noise_test = generate_random_segments(df_noise_test[0].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.55)
df_noise_train = df_noise_train.sample(n=int(len(df_coughs[df_coughs['split'] == 'train']) * 0.15))
df_noise_test = df_noise_test.sample(n=int(len(df_coughs[df_coughs['split'] == 'test']) * 0.15))

df_speech_train = pd.read_csv("/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/metadata.csv.train")
df_speech_test = pd.read_csv("/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/metadata.csv.dev")
df_speech_train["filepath"] = "/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/" + df_speech_train["filepath"]
df_speech_test["filepath"] = "/run/media/fourier/Data1/Pras/Interspeech2025/Datasets/MSPodcast/" + df_speech_test["filepath"]
df_speech_train = generate_random_segments(df_speech_train['filepath'].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.55)
df_speech_test = generate_random_segments(df_speech_test['filepath'].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.55)
df_speech_train = df_speech_train.sample(n=int(len(df_coughs[df_coughs['split'] == 'train']) * 0.35))
df_speech_test = df_speech_test.sample(n=int(len(df_coughs[df_coughs['split'] == 'test']) * 0.35))

100%|██████████| 25900/25900 [02:39<00:00, 162.02it/s]


In [34]:
df_cough_train = df_cough_train.reset_index(drop=True)
df_cough_test = df_cough_test.reset_index(drop=True)

df_speech_train = df_speech_train.reset_index(drop=True)
df_speech_test = df_speech_test.reset_index(drop=True)

df_noise_train = df_noise_train.reset_index(drop=True)
df_noise_test = df_noise_test.reset_index(drop=True)

df_cough_train['index_start'] = 0
df_cough_train['index_end'] = 8800
df_cough_train['label'] = 1
df_cough_train['source'] = "cough"
df_cough_test['index_start'] = 0
df_cough_test['index_end'] = 8800
df_cough_test['label'] = 1
df_cough_test['source'] = "cough"

df_noise_train['label'] = 0
df_noise_test['label'] = 0
df_speech_train['label'] = 0
df_speech_test['label'] = 0

df_noise_train['source'] = "noise"
df_noise_test['source'] = "noise"
df_speech_train['source'] = "speech"
df_speech_test['source'] = "speech"

df_train = pd.concat([df_cough_train, df_speech_train, df_noise_train], axis=0, ignore_index=True)
df_test = pd.concat([df_cough_test, df_speech_test, df_noise_test], axis=0, ignore_index=True)

df_train = df_train[["path_file", "label", "index_start", "index_end", "source"]]
df_test = df_test[["path_file", "label", "index_start", "index_end", "source"]]

In [36]:
df_train.to_csv(f"/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cough_ssl.csv.train", index=False)
df_test.to_csv(f"/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cough_ssl.csv.test", index=False)

In [3]:
import numpy as np
import torch

In [4]:
eye = np.eye(1)
dse_id = torch.from_numpy(eye[1].astype(np.float32)).unsqueeze(0)

IndexError: index 1 is out of bounds for axis 0 with size 1